# Analyze a run

Everything a finished run left behind: TensorBoard curves, the eval
record, per-track holdout results, and the run's charts.


In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))


## TensorBoard

The rsl-rl runner logs training scalars (reward, losses, episode stats)
as TensorBoard events inside every run directory.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs
# (inline panel; plain-terminal alternative:
#    !tensorboard --logdir runs --port 6006 &
#  then open http://localhost:6006)


## Pick a run


In [ ]:
import glob
import json
import os

records = sorted(glob.glob("runs/**/eval_record.json", recursive=True),
                 key=os.path.getmtime)
assert records, "no finished runs under runs/ yet — train first"
run_dir = os.path.dirname(records[-1])
record = json.load(open(records[-1]))
print("run:", run_dir)
record["metrics"]


## Learning curves and per-track holdout

`eval_history` holds the periodic in-training evals; `holdout` holds the
END-OF-TRAINING per-track results on tracks training never saw.


In [ ]:
import pandas as pd

history = pd.DataFrame(record.get("eval_history", []))
display(history)
holdout = pd.DataFrame(record.get("holdout", {})).T
display(holdout)


The same data as saved chart images (written by `Evaluation(charts=True)`):


In [ ]:
from IPython.display import Image as NBImage, display

for png in sorted(glob.glob(os.path.join(run_dir, "charts", "*.png"))):
    print(png)
    display(NBImage(png, width=640))


## Watch the trained policy

Renders a spectator + onboard video of the checkpoint under nominal
conditions (GPU; a few minutes).


In [ ]:
from deepracer_genesis.experiment.visualize import rollout_video

# pass the experiment CLASS the run was trained from, e.g.:
# from examples.camera import CameraZoo
# video = rollout_video(CameraZoo, steps=400)
# print(video)


## What the policy actually saw

`logs/zoo/` keeps the car-view contact sheets from `watch` sessions, and
`logs/dr_editor/` the DR sweeps/grids — `python -m
deepracer_genesis.tools.dr_editor stages --target <your experiment>`
shows the full raw → augmented → stacked pipeline for this run's DR
config.
